# Mission Partie 1 — HR Attrition

Objectif des 3 premières étapes :

1. Préparer l'environnement projet.
2. Charger, comprendre, nettoyer et joindre les 3 fichiers.
3. Construire `X` et `y` prêts pour une future modélisation sklearn.

## Étape 1 — Imports et chargement des fichiers

Place les 3 CSV dans un dossier `data/` à la racine du projet.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_DIR = Path("../data")

sirh_path = DATA_DIR / "extrait_sirh.csv"
eval_path = DATA_DIR / "extrait_eval.csv"
sondage_path = DATA_DIR / "extrait_sondage.csv"

sirh = pd.read_csv(sirh_path)
evaluations = pd.read_csv(eval_path)
sondage = pd.read_csv(sondage_path)

print("SIRH :", sirh.shape)
print("Évaluations :", evaluations.shape)
print("Sondage :", sondage.shape)

## Étape 2 — Compréhension rapide des fichiers

In [ ]:
def audit_dataframe(df: pd.DataFrame, name: str) -> None:
    print(f"\n===== {name} =====")
    display(df.head())
    display(pd.DataFrame({
        "dtype": df.dtypes,
        "nb_valeurs_manquantes": df.isna().sum(),
        "taux_valeurs_manquantes": (df.isna().mean() * 100).round(2),
        "nb_modalites": df.nunique(dropna=False)
    }))
    print("Doublons :", df.duplicated().sum())

audit_dataframe(sirh, "SIRH")
audit_dataframe(evaluations, "Évaluations")
audit_dataframe(sondage, "Sondage")

### Nettoyage simple des colonnes

On corrige surtout :
- `eval_number` pour récupérer l'identifiant numérique de l'employé ;
- `augementation_salaire_precedente` pour obtenir un nombre ;
- quelques variables Oui/Non ou Y en variables binaires.

In [ ]:
def clean_percentage(value):
    """Transforme une valeur comme '11 %' en 11.0."""
    if pd.isna(value):
        return np.nan
    return float(str(value).replace("%", "").strip())

sirh_clean = sirh.copy()
eval_clean = evaluations.copy()
sondage_clean = sondage.copy()

# Clé de jointure extraite depuis E_1, E_2, etc.
eval_clean["id_employee"] = (
    eval_clean["eval_number"]
    .astype(str)
    .str.extract(r"(\d+)")
    .astype(int)
)

# Pourcentage d'augmentation salariale
eval_clean["augmentation_salaire_precedente_pct"] = (
    eval_clean["augementation_salaire_precedente"]
    .apply(clean_percentage)
)

# Variables binaires
eval_clean["heure_supplementaires"] = eval_clean["heure_supplementaires"].map({"Oui": 1, "Non": 0})
sondage_clean["a_quitte_l_entreprise"] = sondage_clean["a_quitte_l_entreprise"].map({"Oui": 1, "Non": 0})
sondage_clean["ayant_enfants"] = sondage_clean["ayant_enfants"].map({"Y": 1, "N": 0})

# Renommer la clé sondage
sondage_clean = sondage_clean.rename(columns={"code_sondage": "id_employee"})

display(eval_clean.head())
display(sondage_clean.head())

### Jointure des 3 fichiers

Les 3 fichiers contiennent les mêmes employés. On peut donc utiliser une jointure `inner` pour garder uniquement les employés présents dans les 3 sources.

In [ ]:
df = (
    sirh_clean
    .merge(sondage_clean, on="id_employee", how="inner")
    .merge(eval_clean, on="id_employee", how="inner")
)

print("Dimensions du DataFrame central :", df.shape)
display(df.head())

# Contrôle de la cible
display(df["a_quitte_l_entreprise"].value_counts())
display((df["a_quitte_l_entreprise"].value_counts(normalize=True) * 100).round(2))

### Statistiques descriptives pour comparer les démissionnaires et les employés encore présents

In [ ]:
target_col = "a_quitte_l_entreprise"

num_cols = df.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c != target_col]

stats_quanti = df.groupby(target_col)[num_cols].mean().T
stats_quanti.columns = ["moyenne_non_quitte", "moyenne_quitte"]
stats_quanti["ecart_quitte_moins_non"] = stats_quanti["moyenne_quitte"] - stats_quanti["moyenne_non_quitte"]
stats_quanti["ecart_absolu"] = stats_quanti["ecart_quitte_moins_non"].abs()

display(stats_quanti.sort_values("ecart_absolu", ascending=False).head(20))

In [ ]:
cat_cols = df.select_dtypes(include="object").columns.tolist()

for col in cat_cols:
    if col in ["eval_number", "augementation_salaire_precedente"]:
        continue

    table = (
        df.groupby(col)[target_col]
        .agg(taux_attrition="mean", effectif="count")
        .sort_values("taux_attrition", ascending=False)
    )
    table["taux_attrition"] = (table["taux_attrition"] * 100).round(1)
    print(f"\nVariable : {col}")
    display(table)

### Visualisations EDA

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x=target_col)
plt.title("Répartition de la cible : a quitté l'entreprise")
plt.xlabel("A quitté l'entreprise")
plt.ylabel("Nombre d'employés")
plt.show()

In [ ]:
variables_quanti_interessantes = [
    "age",
    "revenu_mensuel",
    "annee_experience_totale",
    "annees_dans_l_entreprise",
    "distance_domicile_travail",
    "annees_dans_le_poste_actuel",
    "satisfaction_employee_environnement",
    "satisfaction_employee_equilibre_pro_perso",
]

for col in variables_quanti_interessantes:
    if col in df.columns:
        plt.figure(figsize=(7, 4))
        sns.boxplot(data=df, x=target_col, y=col)
        plt.title(f"{col} selon l'attrition")
        plt.xlabel("A quitté l'entreprise")
        plt.ylabel(col)
        plt.show()

In [ ]:
variables_quali_interessantes = [
    "poste",
    "departement",
    "statut_marital",
    "frequence_deplacement",
    "heure_supplementaires",
    "domaine_etude",
]

for col in variables_quali_interessantes:
    if col in df.columns:
        taux = (
            df.groupby(col)[target_col]
            .mean()
            .sort_values(ascending=False)
            .mul(100)
        )
        plt.figure(figsize=(9, 4))
        sns.barplot(x=taux.index, y=taux.values)
        plt.title(f"Taux d'attrition par {col}")
        plt.ylabel("Taux d'attrition (%)")
        plt.xlabel(col)
        plt.xticks(rotation=45, ha="right")
        plt.show()

## Étape 3 — Préparation de `X` et `y` pour sklearn

Objectif : obtenir :
- `y` : la cible `a_quitte_l_entreprise` ;
- `X` : un DataFrame numérique, sans identifiants inutiles, sans colonne cible, sans variable constante, et encodé.

In [ ]:
def build_modeling_dataset(df: pd.DataFrame):
    df_model = df.copy()

    target = "a_quitte_l_entreprise"

    # Colonnes à supprimer : cible, identifiants, colonnes redondantes ou brutes après nettoyage
    cols_to_drop = [
        target,
        "id_employee",
        "eval_number",
        "augementation_salaire_precedente",
    ]
    cols_to_drop = [c for c in cols_to_drop if c in df_model.columns]

    y = df_model[target].astype(int)
    X = df_model.drop(columns=cols_to_drop)

    # Supprimer les variables constantes, car elles n'apportent aucune information au modèle
    constant_cols = [col for col in X.columns if X[col].nunique(dropna=False) <= 1]
    X = X.drop(columns=constant_cols)
    print("Colonnes constantes supprimées :", constant_cols)

    # Séparer colonnes quantitatives et qualitatives
    numeric_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

    # Encodage OneHot des variables qualitatives nominales
    X_encoded = pd.get_dummies(
        X,
        columns=categorical_features,
        drop_first=True,
        dtype=int
    )

    # Sécurité : remplacer d'éventuelles valeurs manquantes restantes
    X_encoded = X_encoded.fillna(X_encoded.median(numeric_only=True))

    return X_encoded, y

X, y = build_modeling_dataset(df)

print("X :", X.shape)
print("y :", y.shape)
display(X.head())
display(y.head())

### Analyse des corrélations entre variables numériques

In [ ]:
corr = X.corr(method="pearson")

plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Matrice de corrélation de Pearson sur X encodé")
plt.show()

# Repérer les corrélations très fortes
threshold = 0.90
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "correlation"})
)
high_corr_pairs = high_corr_pairs[high_corr_pairs["correlation"].abs() >= threshold]
display(high_corr_pairs.sort_values("correlation", key=lambda s: s.abs(), ascending=False))

In [ ]:
# Option : supprimer automatiquement une des deux variables si corrélation > 0.90
features_to_drop_corr = [
    column for column in upper.columns 
    if any(upper[column].abs() > threshold)
]

print("Features supprimables car trop corrélées :", features_to_drop_corr)

X_reduced = X.drop(columns=features_to_drop_corr)

print("X réduit :", X_reduced.shape)
display(X_reduced.head())

## Premières observations à présenter à Amandine

À compléter avec tes graphiques, mais les premiers résultats à vérifier sont :

- Le taux d'attrition global est d'environ 16 %.
- Les employés ayant quitté l'entreprise semblent avoir en moyenne un revenu mensuel plus bas.
- Ils semblent aussi être plus jeunes, avec moins d'expérience totale et moins d'ancienneté dans l'entreprise.
- Les heures supplémentaires sont associées à un taux d'attrition plus élevé.
- Les déplacements fréquents semblent aussi associés à un taux d'attrition plus élevé.
- Certains postes commerciaux ou de consultant semblent plus exposés que les postes plus seniors ou techniques.

Attention : ces constats sont descriptifs. Ils ne prouvent pas encore une causalité. La modélisation et SHAP permettront ensuite d'identifier les variables les plus explicatives.